# Dört model ailesi: ilk eğitim ve üçlü consensus eğitiminin istatistiksel karşılaştırması

Bu notebook yalnızca mevcut ham tahmin dosyalarını analiz eder. Model eğitmez, ağırlık değiştirmez, test verisini filtrelemez/değiştirmez ve eksik tahmin veya etiket doldurmaz. RoBERTa, BERT, DistilBERT ve hedef veri setine ince ayarlı FinBERT için üç veri setinde toplam 12 karşılaştırma yapar.

McNemar testi eşleştirilmiş doğruluk farkını test eder; Macro-F1 farkı için eşleştirilmiş bootstrap güven aralığı kullanılır. 12 McNemar p-değerinin tamamına birlikte Holm düzeltmesi uygulanır.

In [1]:
# Bağımlılıklar: pandas numpy scikit-learn scipy
# Kurulum: python -m pip install pandas numpy scikit-learn scipy
from pathlib import Path
import random
import sys
import numpy as np
import pandas as pd
from scipy.stats import binomtest

SEED = 20261105
N_BOOT = 10_000
CONFIDENCE_LEVEL = 0.95
ALPHA = 0.05
EXPECTED_N = {'S&P 500': 4510, 'Reuters': 3848, 'Sentetik': 3000}
MODELS = {'RoBERTa': 'roberta', 'BERT': 'bert', 'DistilBERT': 'distilbert', 'FinBERT (hedefe ince ayarlı)': 'finbert'}
DATASETS = {'S&P 500': 'sp500_consensus_external', 'Reuters': 'reuters_consensus_external', 'Sentetik': 'synthetic_external'}
np.random.seed(SEED); random.seed(SEED)
if hasattr(sys.stdout, 'reconfigure'): sys.stdout.reconfigure(encoding='utf-8')

ROOT = Path.cwd()
if ROOT.name == 'app': ROOT = ROOT.parent
if not (ROOT / 'outputs').exists() and (ROOT.parent / 'outputs').exists(): ROOT = ROOT.parent
PRED_ROOT = ROOT / 'outputs' / 'finetuned_model_results' / 'all_models_consensus_external_evaluation'

def prediction_path(training_type, model_dir, dataset_dir):
    return PRED_ROOT / training_type / model_dir / dataset_dir / 'predictions.csv'

def read_predictions(path):
    if not path.exists(): raise FileNotFoundError(f'Gerekli dosya bulunamadı: {path}')
    df = pd.read_csv(path)
    required = {'sample_id', 'gold_label', 'predicted_label'}
    missing = required - set(df.columns)
    if missing: raise ValueError(f'{path} içinde eksik sütun(lar): {sorted(missing)}')
    if df[['sample_id', 'gold_label', 'predicted_label']].isna().any().any():
        raise ValueError(f'{path} içinde sample_id/gold_label/predicted_label eksik değer içeriyor')
    return df

def pairing_report(initial, consensus):
    issues = []
    if initial['sample_id'].duplicated().any(): issues.append('initial sample_id yineleniyor')
    if consensus['sample_id'].duplicated().any(): issues.append('consensus sample_id yineleniyor')
    if set(initial['sample_id']) != set(consensus['sample_id']): issues.append('sample_id kümeleri farklı')
    if initial['sample_id'].tolist() != consensus['sample_id'].tolist(): issues.append('sample_id sırası farklı')
    if len(initial) != len(consensus): issues.append(f'örnek sayısı farklı: {len(initial)} / {len(consensus)}')
    if not initial['gold_label'].astype(str).equals(consensus['gold_label'].astype(str)): issues.append('gold_label değerleri veya sırası farklı')
    return issues

def exact_mcnemar(y, pi, pc):
    i_ok = np.asarray(pi) == np.asarray(y); c_ok = np.asarray(pc) == np.asarray(y)
    b = int(np.sum(i_ok & ~c_ok)); c = int(np.sum(~i_ok & c_ok)); d = b + c
    p = 1.0 if d == 0 else float(binomtest(min(b, c), n=d, p=0.5, alternative='two-sided').pvalue)
    return b, c, p

def macro_f1_from_int(y, pred, k):
    cm = np.bincount(y * k + pred, minlength=k*k).reshape(k, k); tp = np.diag(cm)
    denom_p = cm.sum(axis=0); denom_r = cm.sum(axis=1)
    precision = np.divide(tp, denom_p, out=np.zeros(k), where=denom_p != 0); recall = np.divide(tp, denom_r, out=np.zeros(k), where=denom_r != 0)
    f1 = np.divide(2*precision*recall, precision+recall, out=np.zeros(k), where=(precision+recall) != 0)
    return float(f1.mean())

def paired_bootstrap(y, pi, pc, n_boot=N_BOOT, seed=SEED):
    y, pi, pc = map(np.asarray, (y, pi, pc)); labels = np.unique(np.concatenate([y, pi, pc])); to_int = {x:i for i,x in enumerate(labels)}
    y = np.array([to_int[x] for x in y]); pi = np.array([to_int[x] for x in pi]); pc = np.array([to_int[x] for x in pc]); k = len(labels); n = len(y)
    rng = np.random.default_rng(seed); diffs = np.empty(n_boot); f1_i = macro_f1_from_int(y, pi, k); f1_c = macro_f1_from_int(y, pc, k)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n); diffs[b] = macro_f1_from_int(y[idx], pc[idx], k) - macro_f1_from_int(y[idx], pi[idx], k)
    lo, hi = np.quantile(diffs, [(1-CONFIDENCE_LEVEL)/2, 1-(1-CONFIDENCE_LEVEL)/2])
    return f1_i, f1_c, f1_c-f1_i, float(lo), float(hi)

def base_record(model, dataset, model_dir, dataset_dir, status, n=np.nan, issues=''):
    return {'model': model, 'dataset': dataset, 'model_dir': model_dir, 'dataset_dir': dataset_dir, 'status': status, 'n': n, 'expected_n': EXPECTED_N[dataset], 'n_matches_expected': (bool(n == EXPECTED_N[dataset]) if pd.notna(n) else False), 'initial_correct': np.nan, 'initial_incorrect': np.nan, 'consensus_correct': np.nan, 'consensus_incorrect': np.nan, 'initial_macro_f1': np.nan, 'consensus_macro_f1': np.nan, 'macro_f1_difference_consensus_minus_initial': np.nan, 'bootstrap_ci_low_95': np.nan, 'bootstrap_ci_high_95': np.nan, 'bootstrap_repetitions': N_BOOT, 'mcnemar_b_initial_only_correct': np.nan, 'mcnemar_c_consensus_only_correct': np.nan, 'mcnemar_p_value': np.nan, 'mcnemar_p_value_holm_12_tests': np.nan, 'holm_alpha_0_05_significant': False, 'statistical_evaluation': '', 'pairing_issues': issues}

records = []; validation = []
print(f'Kök dizin: {ROOT}'); print(f'Bootstrap: {N_BOOT:,} tekrar | güven düzeyi: {CONFIDENCE_LEVEL:.0%} | seed: {SEED}')
print('FinBERT yolu: initial_trained/finbert; bu, hedef veri setine ince ayarlı FinBERT çıktısıdır.')
for model, model_dir in MODELS.items():
    for dataset, dataset_dir in DATASETS.items():
        ip = prediction_path('initial_trained', model_dir, dataset_dir); cp = prediction_path('consensus_trained', model_dir, dataset_dir)
        try:
            initial, consensus = read_predictions(ip), read_predictions(cp); issues = pairing_report(initial, consensus); n = len(initial)
            validation.append({'model': model, 'dataset': dataset, 'initial_path': str(ip), 'consensus_path': str(cp), 'initial_n': len(initial), 'consensus_n': len(consensus), 'expected_n': EXPECTED_N[dataset], 'sample_id_set_match': len(issues)==0 or 'sample_id kümeleri farklı' not in issues, 'sample_id_order_match': len(issues)==0 or 'sample_id sırası farklı' not in issues, 'gold_label_match': len(issues)==0 or 'gold_label değerleri veya sırası farklı' not in issues, 'pairing_status': 'MATCHED' if not issues else 'MISMATCH', 'issues': ' | '.join(issues)})
            if issues:
                print(f'{model} | {dataset}: EŞLEŞME BAŞARISIZ — test çalıştırılmadı: {"; ".join(issues)}'); records.append(base_record(model, dataset, model_dir, dataset_dir, 'SKIPPED_PAIRING_MISMATCH', n, ' | '.join(issues))); continue
            y = initial.gold_label.astype(str).to_numpy(); pi = initial.predicted_label.astype(str).to_numpy(); pc = consensus.predicted_label.astype(str).to_numpy(); b, c, p = exact_mcnemar(y, pi, pc); f1i, f1c, diff, lo, hi = paired_bootstrap(y, pi, pc, seed=SEED + len(records))
            ni = int(np.sum(pi == y)); nc = int(np.sum(pc == y)); rec = base_record(model, dataset, model_dir, dataset_dir, 'OK', n)
            rec.update({'initial_correct':ni, 'initial_incorrect':n-ni, 'consensus_correct':nc, 'consensus_incorrect':n-nc, 'initial_macro_f1':f1i, 'consensus_macro_f1':f1c, 'macro_f1_difference_consensus_minus_initial':diff, 'bootstrap_ci_low_95':lo, 'bootstrap_ci_high_95':hi, 'mcnemar_b_initial_only_correct':b, 'mcnemar_c_consensus_only_correct':c, 'mcnemar_p_value':p})
            records.append(rec); print(f'{model} | {dataset} | n={n:,} (beklenen {EXPECTED_N[dataset]:,}) | doğru I/C={ni:,}/{nc:,} | F1 farkı={diff:+.6f} | McNemar p={p:.6g} | GA=[{lo:+.6f}, {hi:+.6f}]')
        except (FileNotFoundError, ValueError) as exc:
            print(f'{model} | {dataset}: ANALİZ ÇALIŞTIRILMADI — {exc}'); records.append(base_record(model, dataset, model_dir, dataset_dir, 'SKIPPED_MISSING_OR_INVALID_FILE', issues=str(exc))); validation.append({'model':model, 'dataset':dataset, 'initial_path':str(ip), 'consensus_path':str(cp), 'initial_n':np.nan, 'consensus_n':np.nan, 'expected_n':EXPECTED_N[dataset], 'sample_id_set_match':False, 'sample_id_order_match':False, 'gold_label_match':False, 'pairing_status':'MISSING_OR_INVALID', 'issues':str(exc)})

results = pd.DataFrame(records); valid = results.status.eq('OK') & results.mcnemar_p_value.notna(); pvals = results.loc[valid, 'mcnemar_p_value'].to_numpy(); order = np.argsort(pvals); adjusted = np.full(len(pvals), np.nan)
if len(pvals): adjusted[order] = np.minimum(np.maximum.accumulate((len(pvals)-np.arange(len(pvals))) * pvals[order]), 1.0)
results.loc[valid, 'mcnemar_p_value_holm_12_tests'] = adjusted; results.loc[valid, 'holm_alpha_0_05_significant'] = adjusted < ALPHA
def evaluation(row):
    if row.status != 'OK': return 'Değerlendirilemedi: eşleşme/dosya sorunu'
    acc_text = 'McNemar doğruluk farkı anlamlı' if row.mcnemar_p_value_holm_12_tests < ALPHA else 'McNemar doğruluk farkı anlamlı değil'
    f1_text = 'bootstrap Macro-F1 farkı için GA sıfırı içermiyor' if row.bootstrap_ci_low_95 > 0 or row.bootstrap_ci_high_95 < 0 else 'bootstrap Macro-F1 farkı için GA sıfırı içeriyor'
    return acc_text + '; ' + f1_text
results['statistical_evaluation'] = results.apply(evaluation, axis=1)

validation_df = pd.DataFrame(validation)
model_summary = results.groupby('model', dropna=False).agg(comparisons=('dataset','count'), completed=('status', lambda s: int((s=='OK').sum())), mean_macro_f1_difference=('macro_f1_difference_consensus_minus_initial','mean'), significant_mcnemar_after_holm=('holm_alpha_0_05_significant','sum')).reset_index()
dataset_summary = results.groupby('dataset', dropna=False).agg(comparisons=('model','count'), completed=('status', lambda s: int((s=='OK').sum())), mean_macro_f1_difference=('macro_f1_difference_consensus_minus_initial','mean'), significant_mcnemar_after_holm=('holm_alpha_0_05_significant','sum')).reset_index()

print('\n' + '='*150); print('12 KARŞILAŞTIRMANIN AYRINTILI SONUÇLARI'); print(results[['model','dataset','n','initial_macro_f1','consensus_macro_f1','macro_f1_difference_consensus_minus_initial','bootstrap_ci_low_95','bootstrap_ci_high_95','mcnemar_p_value','mcnemar_p_value_holm_12_tests','statistical_evaluation']].to_string(index=False)); print('\nModel özeti:'); print(model_summary.to_string(index=False)); print('\nVeri seti özeti:'); print(dataset_summary.to_string(index=False)); print('\nEşleşme doğrulama raporu:')
print(validation_df.to_string(index=False))
print('\nNot: Sonuçlar dosyaya yazılmadı; tüm çıktılar yalnızca terminale yazdırıldı.')

Kök dizin: d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis
Bootstrap: 10,000 tekrar | güven düzeyi: 95% | seed: 20261105
FinBERT yolu: initial_trained/finbert; bu, hedef veri setine ince ayarlı FinBERT çıktısıdır.
RoBERTa | S&P 500 | n=4,510 (beklenen 4,510) | doğru I/C=3,663/3,871 | F1 farkı=+0.047765 | McNemar p=3.83211e-21 | GA=[+0.037434, +0.058009]
RoBERTa | Reuters | n=3,848 (beklenen 3,848) | doğru I/C=2,882/2,937 | F1 farkı=+0.011157 | McNemar p=0.00522576 | GA=[+0.001025, +0.020962]
RoBERTa | Sentetik | n=3,000 (beklenen 3,000) | doğru I/C=2,986/2,986 | F1 farkı=+0.000002 | McNemar p=1 | GA=[-0.002345, +0.002355]
BERT | S&P 500 | n=4,510 (beklenen 4,510) | doğru I/C=3,501/3,666 | F1 farkı=+0.041202 | McNemar p=7.25954e-13 | GA=[+0.030454, +0.051920]
BERT | Reuters | n=3,848 (beklenen 3,848) | doğru I/C=2,688/2,770 | F1 farkı=+0.015854 | McNemar p=0.00012914 | GA=[+0.005112, +0.026684]
BERT | Sentetik | n=3,000 (beklenen 3,000) | doğru I/C=2,923/2,945 | F

## Dosyalar ve kurulum

Tahmin dosyaları `outputs/finetuned_model_results/all_models_consensus_external_evaluation/{initial_trained,consensus_trained}/{roberta,bert,distilbert,finbert}/{dataset}/predictions.csv` yollarında aranır. FinBERT için `initial_trained/finbert` hedef veri setine ince ayarlı modeldir. Notebook yeni CSV üretmez; ayrıntılı karşılaştırma, eşleşme doğrulama ve özet tabloları yalnızca terminale yazdırır.